[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# CLIP

[CLIP](https://github.com/openai/CLIP)（对比语言-图像预训练）是 OpenAI 在一系列（图像，文本）对上训练的神经网络。

可以用自然语言指示它：给定一张图像，预测最相关的文本片段，而不需要直接针对该任务做优化，这和 GPT-2、GPT-3 的零样本能力类似。我们发现 CLIP 在 ImageNet 上以"零样本"方式达到了原始 ResNet50 的性能，没有使用原来的 128 万张带标签样本中的任何一张，克服了计算机视觉中的几个重大挑战。


![](https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png)


In [ ]:
# 如果你使用 google colab，取消下面代码的注释：
#%pip install git+https://github.com/openai/CLIP.git
#%mkdir data
#%cd data
#!wget https://raw.githubusercontent.com/dataflowr/notebooks/master/Module19/data/cat.jpg
#!wget https://raw.githubusercontent.com/dataflowr/notebooks/master/Module19/data/dog.png
#!wget https://raw.githubusercontent.com/dataflowr/notebooks/master/Module19/data/caltech101_full.json
#%cd ..

In [ ]:
import torch
import clip
from PIL import Image
import numpy as np

In [ ]:
dog_image = Image.open("data/dog.png")
cat_image = Image.open("data/cat.jpg")

In [ ]:
dog_image

In [ ]:
cat_image

# 第一次使用 CLIP

使用[代码片段](https://github.com/openai/CLIP#usage)，为上面的 2 张图像得到正确的标签。


注意，提供的代码里没有用到文本和图像的特征。验证一下，这些概率可以直接从特征中恢复出来。


# 用 CLIP 构建分类器

验证下面的分类器对上面的图像是有效的。


In [ ]:
class Classifier_CLIP:
    def __init__(self, labels):
        self.labels = labels
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model, self.preprocess = clip.load("ViT-B/32", device=self.device)
        self.text = clip.tokenize(labels).to(self.device)
        
    def classify(self, image_pil, verbose=False):
        image = self.preprocess(image_pil).unsqueeze(0).to(device)
        with torch.no_grad():
            logits_per_image, logits_per_text = self.model(image, self.text)
            probs = logits_per_image.softmax(dim=-1).cpu().numpy()
            if verbose:
                print('predicted class: ', self.labels[np.argmax(probs)])
        return np.argmax(probs)

In [ ]:
#classifier.classify(dog_image, verbose=True)

In [ ]:
#classifier.classify(cat_image, verbose=True)

# 在 Caltech 101 上测试分类器

现在我们要看看这个分类器在 [Caltech 101](https://data.caltech.edu/records/mzrjq-6wc02) 数据集上的表现。

你首先需要用 [torchvision](https://pytorch.org/vision/stable/generated/torchvision.datasets.Caltech101.html#torchvision.datasets.Caltech101) 下载数据集


In [ ]:
import torchvision
#caltech_data = torchvision.datasets.Caltech101('data/', download=True)

In [ ]:
caltech_data

In [ ]:
k = 4578
caltech_data[k]

In [ ]:
caltech_data[k][0]

In [ ]:
caltech_data.categories[caltech_data[k][1]]

现在，你需要给 `Classifier_CLIP` 类添加方法。你可以看 Sean Osier 的这篇[不错的博客](https://www.seanosier.com/2021/03/20/python-add-method-existing-class/)了解怎么做。

先做一个方法：为标签创建对应的文本并做分词。完成后，检查分类器对上面图像的预测。它正确吗？


现在，添加两个方法：`predict` 接收一批图像，计算对应的概率和预测；`test` 接收一个 dataloader，用 predict 计算分类器在数据集上的准确率。

提示：创建 dataloader 可以用 `from more_itertools import chunked`


# 用 GPT 获得更好的性能！

利用 Sachit Menon 和 Carl Vondrick（ICLR 2023）的 [Visual Classification via Description from Large Language Models](https://github.com/sachit-menon/classify_by_description_release/tree/master#visual-classification-via-description-from-large-language-models) 的想法，尝试获得更好的性能！
![](https://raw.githubusercontent.com/sachit-menon/classify_by_description_release/master/figs/latent-points.png)

如果你不想做提示工程，文件 `caltech101_full.json` 里提供了描述符


In [ ]:
import json

def load_descriptors(self, filename):
    # 打开 JSON 文件
    f = open(filename)
    self.descriptors = json.load(f)
    pass

In [ ]:
Classifier_CLIP.load_descriptors = load_descriptors

In [ ]:
classifier_caltech.load_descriptors('data/caltech101_full.json')

In [ ]:
classifier_caltech.descriptors

# 图像分类的提示工程

尝试获得更好的描述符！


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = # os.environ["OPENAI_API_KEY"] = # 在这里填你的 key

def stringtolist(description):
    return [descriptor[2:] for descriptor in description.split('\n') if (descriptor != '') and (descriptor.startswith('- '))]

我使用[和原论文一样的提示](https://github.com/sachit-menon/classify_by_description_release/blob/master/generate_descriptors.py)，只是适配了新 API。它还可以改进……


In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What are useful visual features for distinguishing a lemur in a photo?"},
    {"role": "assistant", "content": """There are several useful visual features to tell there is a lemur in a photo:
- four-limbed primate
- black, grey, white, brown, or red-brown
- wet and hairless nose with curved nostrils
- long tail
- large eyes
- furry bodies
- clawed hands and feet"""},
    {"role": "user", "content": "What are useful visual features for distinguishing a television in a photo?"},
    {"role": "assistant", "content": """There are several useful visual features to tell there is a television in a photo:
- electronic device
- black or grey
- a large, rectangular screen
- a stand or mount to support the screen
- one or more speakers
- a power cord
- input ports for connecting to other devices
- a remote control"""},
    {"role": "user", "content": "What are useful visual features for distinguishing a dragonfly in a photo? Provide an answer following the above pattern, give only the list of visual features."}
  ]
)

In [ ]:
response.choices[0].message.content

In [ ]:
def generate_prompt(category_name: str):
    # 你可以把例子换成任何你想要的内容；这些是随机挑选的，效果还行，可以改进
    return [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What are useful visual features for distinguishing a lemur in a photo?"},
    {"role": "assistant", "content": """There are several useful visual features to tell there is a lemur in a photo:
- four-limbed primate
- black, grey, white, brown, or red-brown
- wet and hairless nose with curved nostrils
- long tail
- large eyes
- furry bodies
- clawed hands and feet"""},
    {"role": "user", "content": "What are useful visual features for distinguishing a television in a photo?"},
    {"role": "assistant", "content": """There are several useful visual features to tell there is a television in a photo:
- electronic device
- black or grey
- a large, rectangular screen
- a stand or mount to support the screen
- one or more speakers
- a power cord
- input ports for connecting to other devices
- a remote control"""},
    {"role": "user", "content": f"What are useful visual features for distinguishing a {category_name} in a photo? Provide an answer following the above pattern, give only the list of visual features."}
  ]

def obtain_descriptors_and_save(filename, class_list):
    responses = {}
    descriptors = {}
    prompts = [generate_prompt(category.replace('_', ' ')) for category in class_list]
    client = OpenAI()

    responses = [client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages= prompt
    ) for prompt in prompts]
    
    
    response_texts = [resp.choices[0].message.content for resp in responses]
    descriptors_list = [stringtolist(response_text) for response_text in response_texts]
    descriptors = {cat: descr for cat, descr in zip(class_list, descriptors_list)}

    # 把描述符保存到 json 文件
    if not filename.endswith('.json'):
        filename += '.json'
    with open(filename, 'w') as fp:
        json.dump(descriptors, fp)